# TekaRx `gnn-full` on Google Colab

This notebook builds the leakage-controlled FAERS experiment with **2019Q1–2023Q4 for training, 2024Q1 for validation, and 2024Q2 held out for final testing**. It uses Google Drive for durable artifacts and Colab's `/content` disk for DuckDB, memory-mapped graph construction, and training.

> TekaRx outputs are research decision-support signals, not diagnoses or clinical advice. FAERS reports do not establish causality.


## How to run this notebook

Run the work in three restartable stages:

1. **Source staging (CPU runtime):** download and convert immutable sources under Drive.
2. **Cohort and graph build (high-memory CPU runtime):** copy compact inputs to `/content`, build features and the memory-mapped graph, then checkpoint to Drive.
3. **GNN training (GPU runtime):** restore only the graph descriptor and sidecars to `/content`, train with CUDA, and copy the model to Drive.

After a Colab runtime reset, rerun the setup cells before resuming the relevant stage. Do not train directly from the mounted Drive filesystem. Do not add a test-evaluation flag while selecting features or hyperparameters.


## 0. Publish the reviewed code first

The notebook clones GitHub, so the dosage, prospective-split, and memory-mapped graph implementation must already be committed and pushed. Set `GIT_REF` below to the reviewed full commit SHA for a reproducible run; mutable branch refs are rejected. Never paste a GitHub token into this notebook.


In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("This notebook must run in a Google Colab managed runtime.") from exc

drive.mount("/content/drive")


In [ ]:
from __future__ import annotations

import importlib.metadata
import json
import os
import platform
import shlex
import shutil
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

REPO_URL = "https://github.com/matthew-sudo2/Teka-Rx.git"
REPO_DIR = Path("/content/Teka-Rx")
GIT_REF = "d974b6ad582a9731f9968d70e50fbddc090c8d1d"

DRIVE_DATA = Path("/content/drive/MyDrive/Teka-Rx-full/data")
LOCAL_DATA = Path("/content/tekarx-data")
MEMORY_LIMIT = "6GB"
THREADS = 1
MATERIALIZATION_BATCH_SIZE = 131_072
XGB_BATCH_SIZE = 65_536
GNN_BATCH_SIZE = 8_192
EDGE_CHUNK_SIZE = 250_000
GNN_SEED = 42
BUILD_DAILYMED = True
MIN_DRIVE_FREE_GIB = 35
RECOMMENDED_DRIVE_FREE_GIB = 80
LOCAL_BUILD_HEADROOM_GIB = 45

SPLITS = {
    "train": [f"{year}Q{quarter}" for year in range(2019, 2024) for quarter in range(1, 5)],
    "validation": ["2024Q1"],
    "test": ["2024Q2"],
}
EXPECTED_QUARTERS = tuple(quarter for values in SPLITS.values() for quarter in values)
FAERS_TABLES = ("demo", "drug", "indi", "reac", "outc", "delete")
BUILD_STAGE_ARTIFACTS = {
    "cohort": (
        "tekarx_cohort.parquet", "case_splits.parquet", "cohort_manifest.json",
        "splits/faers_gnn-full.json", "edges/report_drug.parquet",
        "edges/report_reaction.parquet", "edges/report_outcome.parquet",
    ),
    "dictionary": ("drug_dictionary.parquet", "drug_dictionary_manifest.json"),
    "tabular": (
        "tekarx_cohort_enriched.parquet", "drug_risk_lookup.parquet",
        "dose_normalization_lookup.parquet", "edges/report_drug_dose.parquet",
        "cohort_enriched_manifest.json",
    ),
    "rescue": (
        "tekarx_cohort_feature_rescue.parquet", "high_risk_drug_pairs.parquet",
        "indication_lookup.parquet", "feature_rescue_manifest.json",
    ),
}
BUILD_STAGE_MARKERS = {
    stage: f"_{stage.upper()}_SUCCESS.json" for stage in BUILD_STAGE_ARTIFACTS
}

if len(EXPECTED_QUARTERS) != 22 or len(set(EXPECTED_QUARTERS)) != 22:
    raise RuntimeError(f"The frozen gnn-full plan must contain 22 unique quarters: {EXPECTED_QUARTERS}")
if sys.version_info < (3, 12):
    raise RuntimeError(f"TekaRx requires Python 3.12 or newer; found {sys.version}.")
if DRIVE_DATA == LOCAL_DATA or not str(LOCAL_DATA).startswith("/content/"):
    raise RuntimeError("LOCAL_DATA must be a separate directory on Colab's /content disk.")

DRIVE_DATA.mkdir(parents=True, exist_ok=True)
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
print(f"Durable data: {DRIVE_DATA}")
print(f"Fast scratch: {LOCAL_DATA}")


In [ ]:
def gibibytes(value: int) -> float:
    return value / (1024**3)


def run_command(*parts: object, cwd: Path | None = None) -> subprocess.CompletedProcess[str]:
    command = [str(part) for part in parts]
    print("$", shlex.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True, text=True)


def run_tekarx(*parts: object, data_dir: Path) -> subprocess.CompletedProcess[str]:
    return run_command("tekarx", *parts, "--data-dir", data_dir)


def load_json(path: Path) -> dict[str, object]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json_atomic(path: Path, payload: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def tree_inventory(root: Path) -> dict[str, int]:
    if not root.exists():
        return {}
    if root.is_file():
        return {root.name: root.stat().st_size}
    return {
        path.relative_to(root).as_posix(): path.stat().st_size
        for path in root.rglob("*")
        if path.is_file()
    }


def tree_size(root: Path) -> int:
    return sum(tree_inventory(root).values())


def copy_tree_verified(source: Path, destination: Path, *, label: str) -> None:
    if not source.is_dir():
        raise FileNotFoundError(f"Missing {label}: {source}")
    source_inventory = tree_inventory(source)
    pending = [
        (relative, size) for relative, size in source_inventory.items()
        if not (destination / relative).is_file()
        or (destination / relative).stat().st_size != size
    ]
    pending_bytes = sum(size for _, size in pending)
    print(
        f"Copying {label}: {len(pending):,}/{len(source_inventory):,} files pending, "
        f"{gibibytes(pending_bytes):.2f} GiB", flush=True
    )
    copied_bytes = 0
    for index, (relative, size) in enumerate(sorted(pending), start=1):
        copy_file_verified(
            source / relative, destination / relative, label=f"{label}: {relative}"
        )
        copied_bytes += size
        if index == 1 or index % 10 == 0 or index == len(pending):
            print(
                f"  {label}: {index:,}/{len(pending):,} files, "
                f"{gibibytes(copied_bytes):.2f}/{gibibytes(pending_bytes):.2f} GiB",
                flush=True,
            )
    mismatches = [
        relative
        for relative, size in source_inventory.items()
        if not (destination / relative).is_file() or (destination / relative).stat().st_size != size
    ]
    if mismatches:
        raise RuntimeError(f"Incomplete {label} copy; first mismatches: {mismatches[:5]}")
    print(f"Verified {label} copy.")


def copy_file_verified(source: Path, destination: Path, *, label: str) -> None:
    if not source.is_file():
        raise FileNotFoundError(f"Missing {label}: {source}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".uploading")
    shutil.copy2(source, temporary)
    if temporary.stat().st_size != source.stat().st_size:
        raise RuntimeError(f"Size mismatch while copying {label}")
    os.replace(temporary, destination)


def assert_faers_scope(root: Path, *, require_complete: bool) -> None:
    expected = set(EXPECTED_QUARTERS)
    problems: list[str] = []
    for table in FAERS_TABLES:
        directory = root / "interim" / "faers" / table
        present = {path.stem for path in directory.glob("*.parquet")} if directory.exists() else set()
        extra = sorted(present - expected)
        missing = sorted(expected - present) if require_complete else []
        if extra:
            problems.append(f"{table}: unexpected quarters {extra}")
        if missing:
            problems.append(f"{table}: missing quarters {missing}")
    raw_root = root / "raw" / "faers"
    if raw_root.exists():
        raw_quarters = {path.name for path in raw_root.iterdir() if path.is_dir() and path.name[:2] == "20"}
        extra_raw = sorted(raw_quarters - expected)
        if extra_raw:
            problems.append(f"raw FAERS: unexpected quarters {extra_raw}")
    if problems:
        raise RuntimeError("FAERS experiment-root check failed:\n  - " + "\n  - ".join(problems))


def copy_delta_bytes(source: Path, destination: Path) -> int:
    if source.is_file():
        return 0 if destination.is_file() and destination.stat().st_size == source.stat().st_size else source.stat().st_size
    source_inventory = tree_inventory(source)
    return sum(
        size for relative, size in source_inventory.items()
        if not (destination / relative).is_file() or (destination / relative).stat().st_size != size
    )


def require_local_capacity(
    copy_pairs: list[tuple[Path, Path]], *, headroom_gib: float
) -> None:
    source_bytes = sum(copy_delta_bytes(source, destination) for source, destination in copy_pairs)
    free_bytes = shutil.disk_usage("/content").free
    required_bytes = source_bytes + int(headroom_gib * 1024**3)
    print(
        f"Incremental local copy: {gibibytes(source_bytes):.2f} GiB; "
        f"free: {gibibytes(free_bytes):.2f} GiB; required with headroom: "
        f"{gibibytes(required_bytes):.2f} GiB"
    )
    if free_bytes < required_bytes:
        raise RuntimeError("Insufficient /content space. Choose a larger runtime disk before continuing.")


In [ ]:
local_usage = shutil.disk_usage("/content")
drive_usage = shutil.disk_usage(DRIVE_DATA)
page_size = os.sysconf("SC_PAGE_SIZE")
physical_pages = os.sysconf("SC_PHYS_PAGES")
memory_gib = gibibytes(page_size * physical_pages)

print(f"Python: {platform.python_version()}")
print(f"System RAM: {memory_gib:.2f} GiB")
print(f"/content free: {gibibytes(local_usage.free):.2f} GiB")
drive_free_gib = gibibytes(drive_usage.free)
print(f"Drive free: {drive_free_gib:.2f} GiB")
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi is None:
    print("GPU utility not present: this is a CPU runtime.")
else:
    subprocess.run([nvidia_smi], check=False, text=True)

if drive_free_gib < MIN_DRIVE_FREE_GIB:
    raise RuntimeError(
        f"At least {MIN_DRIVE_FREE_GIB} GiB of free Drive space is required; "
        f"found {drive_free_gib:.2f} GiB."
    )
if drive_free_gib < RECOMMENDED_DRIVE_FREE_GIB:
    print(
        f"WARNING: {drive_free_gib:.2f} GiB is below the recommended "
        f"{RECOMMENDED_DRIVE_FREE_GIB} GiB reserve, but above the supported minimum. "
        "Keep only this experiment in the Teka-Rx-full root and monitor space after each stage."
    )
if memory_gib < 12:
    print("WARNING: choose a high-memory runtime before Stage 2 if one is available.")


In [ ]:
if GIT_REF.startswith("REPLACE_") or GIT_REF in {"main", "origin/main"}:
    raise RuntimeError("Set GIT_REF to the reviewed immutable 40-character commit SHA.")
if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
    backup = REPO_DIR.with_name(
        f"{REPO_DIR.name}-incomplete-{datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')}"
    )
    shutil.move(REPO_DIR, backup)
    print(f"Moved incomplete clone to {backup}")
if not (REPO_DIR / ".git").is_dir():
    run_command("git", "clone", REPO_URL, REPO_DIR)
run_command("git", "-C", REPO_DIR, "fetch", "--prune", "origin")
run_command("git", "-C", REPO_DIR, "checkout", "--detach", GIT_REF)

required_implementation = (
    REPO_DIR / "src" / "tekarx" / "transform" / "dosage.py",
    REPO_DIR / "src" / "tekarx" / "transform" / "graph_storage.py",
    REPO_DIR / "src" / "tekarx" / "modeling" / "gnn.py",
)
missing_code = [str(path) for path in required_implementation if not path.is_file()]
if missing_code:
    raise RuntimeError(
        "The selected Git ref is the old repository state. Commit and push the reviewed "
        f"implementation first. Missing: {missing_code}"
    )

RESOLVED_GIT_SHA = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
run_command(sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[dev,graph,notebook]")
repo_source = str((REPO_DIR / "src").resolve())
if repo_source not in sys.path:
    sys.path.insert(0, repo_source)
importlib.invalidate_caches()
import tekarx
print(f"Kernel package: {Path(tekarx.__file__).resolve()}")
print(f"Resolved code revision: {RESOLVED_GIT_SHA}")


In [ ]:
import duckdb
import numpy as np
import pyarrow
import torch
import torch_geometric
import xgboost

versions = {
    "python": platform.python_version(),
    "tekarx": importlib.metadata.version("tekarx"),
    "duckdb": duckdb.__version__,
    "numpy": np.__version__,
    "pyarrow": pyarrow.__version__,
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "xgboost": xgboost.__version__,
}
print(json.dumps(versions, indent=2))
print(f"CUDA available in this runtime: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
run_command(sys.executable, "-m", "pip", "check")
run_command(sys.executable, "-m", "ruff", "check", ".", cwd=REPO_DIR)
run_command(sys.executable, "-m", "pytest", "-q", cwd=REPO_DIR)
for command in (
    "build-cohort", "build-drug-dictionary", "add-tabular-features",
    "feature-rescue", "build-graph", "train-gnn",
):
    run_command("tekarx", command, "--help")


## 1. Stage immutable sources in Drive

A standard CPU runtime is enough for this stage. The dedicated `Teka-Rx-full` root prevents later FAERS quarters from changing latest-case-version selection. Completed downloads and Parquet tables are checksum-cached, so rerunning these cells reuses verified artifacts. If Colab stops during ZIP extraction, inspect the exact incomplete quarter directory before retrying; the extractor deliberately refuses to overwrite it.


In [ ]:
experiment_plan = {
    "name": "gnn-full",
    "splits": SPLITS,
    "case_group_key": "caseid",
    "latest_version_key": "caseversion",
    "test_policy": "locked until model and hyperparameters are frozen",
}
plan_path = DRIVE_DATA / "experiment_definition.json"
if plan_path.is_file():
    existing_plan = load_json(plan_path)
    if existing_plan != experiment_plan:
        raise RuntimeError(f"Conflicting experiment definition already exists: {plan_path}")
else:
    write_json_atomic(plan_path, experiment_plan)
assert_faers_scope(DRIVE_DATA, require_complete=False)
source_marker_path = DRIVE_DATA / "_SOURCES_SUCCESS.json"
source_marker = load_json(source_marker_path) if source_marker_path.is_file() else {}
expected_source_files = [
    DRIVE_DATA / "interim" / "faers" / table / f"{quarter}.parquet"
    for table in FAERS_TABLES for quarter in EXPECTED_QUARTERS
]
expected_source_files.extend(
    DRIVE_DATA / "interim" / "drugcentral" / name
    for name in ("structures.parquet", "synonyms.parquet", "struct2atc.parquet")
)
if BUILD_DAILYMED:
    expected_source_files.extend(
        DRIVE_DATA / "interim" / "dailymed" / name
        for name in ("rxnorm.parquet", "pharmacologic-class.parquet", "metadata.parquet")
    )
SOURCES_ALREADY_COMPLETE = (
    source_marker.get("stage") == "sources_complete"
    and source_marker.get("split_preset") == "gnn-full"
    and source_marker.get("quarters") == list(EXPECTED_QUARTERS)
    and all(path.is_file() for path in expected_source_files)
)
print(json.dumps(experiment_plan, indent=2))
print(f"Verified source checkpoint available: {SOURCES_ALREADY_COMPLETE}")


In [ ]:
if SOURCES_ALREADY_COMPLETE:
    print("SKIP DrugCentral staging: verified source checkpoint is present.")
else:
    run_tekarx("extract-drugcentral", data_dir=DRIVE_DATA)
    run_tekarx("build-drugcentral", data_dir=DRIVE_DATA)


In [ ]:
import time

from tqdm.auto import tqdm

assert_faers_scope(DRIVE_DATA, require_complete=False)

if SOURCES_ALREADY_COMPLETE:
    print("SKIP FAERS staging: verified 22-quarter source checkpoint is present.")
else:
    with tqdm(total=len(EXPECTED_QUARTERS), desc="FAERS quarters", unit="quarter") as overall:
        for position, quarter in enumerate(EXPECTED_QUARTERS, start=1):
            started = time.monotonic()
            drive_free = gibibytes(shutil.disk_usage(DRIVE_DATA).free)
            if drive_free < MIN_DRIVE_FREE_GIB:
                raise RuntimeError(f"Drive has only {drive_free:.2f} GiB free before {quarter}.")
            archive = DRIVE_DATA / "raw" / "faers" / quarter / "source.zip"
            extraction_marker = (
                DRIVE_DATA / "raw" / "faers" / quarter / "extracted" / ".complete"
            )
            expected_parquet = [
                DRIVE_DATA / "interim" / "faers" / table / f"{quarter}.parquet"
                for table in FAERS_TABLES
            ]
            cached_tables = sum(path.is_file() for path in expected_parquet)
            archive_mb = archive.stat().st_size / 1024**2 if archive.is_file() else 0.0
            tqdm.write(
                f"[{position:02d}/{len(EXPECTED_QUARTERS)}] {quarter} starting | "
                f"archive={archive_mb:,.1f} MiB | extracted={extraction_marker.is_file()} | "
                f"parquet={cached_tables}/{len(FAERS_TABLES)}"
            )

            overall.set_postfix_str(f"{quarter}: download/checksum")
            run_tekarx("extract-faers", "--quarter", quarter, data_dir=DRIVE_DATA)

            overall.set_postfix_str(f"{quarter}: Parquet")
            run_tekarx("build-faers", "--quarter", quarter, data_dir=DRIVE_DATA)

            missing = [path for path in expected_parquet if not path.is_file()]
            if missing:
                raise RuntimeError(f"{quarter} did not produce all FAERS tables: {missing}")
            elapsed_minutes = (time.monotonic() - started) / 60
            tqdm.write(
                f"[{position:02d}/{len(EXPECTED_QUARTERS)}] {quarter} complete in "
                f"{elapsed_minutes:.1f} min ({len(expected_parquet)} Parquet tables)"
            )
            overall.update(1)

    assert_faers_scope(DRIVE_DATA, require_complete=True)
    print("All required FAERS quarters are downloaded, extracted, and built.")


In [ ]:
if BUILD_DAILYMED and SOURCES_ALREADY_COMPLETE:
    print("SKIP DailyMed/RxNorm staging: verified source checkpoint is present.")
elif BUILD_DAILYMED:
    run_tekarx("extract-dailymed", data_dir=DRIVE_DATA)
    run_tekarx("build-dailymed", data_dir=DRIVE_DATA)
    run_tekarx("build-rxnorm-lookup", data_dir=DRIVE_DATA)
else:
    print("DailyMed/RxNorm staging skipped. DrugCentral exact and synonym mapping remains available.")


In [ ]:
import pandas as pd
import pyarrow.parquet as pq

assert_faers_scope(DRIVE_DATA, require_complete=True)

def audit_snappy_parquet(path: Path, *, allow_empty: bool = False) -> pq.ParquetFile:
    parquet = pq.ParquetFile(path)
    if parquet.metadata.num_rows == 0 and not allow_empty:
        raise RuntimeError(f"Unexpected empty Parquet input: {path}")
    for row_group in range(parquet.metadata.num_row_groups):
        for column in range(parquet.metadata.num_columns):
            compression = parquet.metadata.row_group(row_group).column(column).compression
            if compression != "SNAPPY":
                raise RuntimeError(f"Non-Snappy source artifact: {path}")
    return parquet


audit_rows = []
for table in FAERS_TABLES:
    paths = sorted((DRIVE_DATA / "interim" / "faers" / table).glob("*.parquet"))
    parquets = [audit_snappy_parquet(path, allow_empty=(table == "delete")) for path in paths]
    audit_rows.append(
        {
            "table": table,
            "quarters": len(paths),
            "rows": sum(parquet.metadata.num_rows for parquet in parquets),
            "size_gib": gibibytes(sum(path.stat().st_size for path in paths)),
        }
    )
display(pd.DataFrame(audit_rows))

required_drugcentral = ("structures.parquet", "synonyms.parquet", "struct2atc.parquet")
missing_drugcentral = [
    name
    for name in required_drugcentral
    if not (DRIVE_DATA / "interim" / "drugcentral" / name).is_file()
]
drugcentral_dumps = sorted((DRIVE_DATA / "raw" / "drugcentral").glob("*.sql.gz"))
if missing_drugcentral or len(drugcentral_dumps) != 1:
    raise RuntimeError(
        f"DrugCentral preflight failed: missing={missing_drugcentral}, dumps={drugcentral_dumps}"
    )
for name in required_drugcentral:
    audit_snappy_parquet(DRIVE_DATA / "interim" / "drugcentral" / name)
if BUILD_DAILYMED:
    reference_inputs = (
        DRIVE_DATA / "interim/dailymed/rxnorm.parquet",
        DRIVE_DATA / "interim/dailymed/pharmacologic-class.parquet",
        DRIVE_DATA / "interim/dailymed/metadata.parquet",
        DRIVE_DATA / "interim/dailymed/manifest.json",
        DRIVE_DATA / "interim/drugcentral/rxnorm_lookup.parquet",
        DRIVE_DATA / "interim/drugcentral/rxnorm_lookup_manifest.json",
    )
    missing_references = [str(path) for path in reference_inputs if not path.is_file()]
    if missing_references:
        raise RuntimeError(f"DailyMed/RxNorm preflight failed: {missing_references}")
    for path in reference_inputs:
        if path.suffix == ".parquet":
            audit_snappy_parquet(path, allow_empty=(path.name == "rxnorm_lookup.parquet"))
source_artifacts = [*expected_source_files, drugcentral_dumps[0]]
source_success = {
    "stage": "sources_complete",
    "completed_at_utc": datetime.now(UTC).isoformat(),
    "git_sha": RESOLVED_GIT_SHA,
    "split_preset": "gnn-full",
    "quarters": list(EXPECTED_QUARTERS),
    "artifacts": {
        path.relative_to(DRIVE_DATA).as_posix(): path.stat().st_size
        for path in source_artifacts
    },
}
write_json_atomic(DRIVE_DATA / "_SOURCES_SUCCESS.json", source_success)
print("Stage 1 complete: all 22 quarters x 6 FAERS tables and reference inputs verified.")


## 2. Build the cohort, features, and graph on local SSD

Use a high-memory CPU runtime if available, then rerun the setup cells. This stage copies only interim Parquet/reference files and the one DrugCentral SQL dump to `/content`; raw FAERS ZIP/TXT files remain in Drive. Feature artifacts are checkpointed before the graph build so a later runtime can resume from Drive.


In [ ]:
assert_faers_scope(DRIVE_DATA, require_complete=True)
source_success_path = DRIVE_DATA / "_SOURCES_SUCCESS.json"
if not source_success_path.is_file():
    raise RuntimeError("Source checkpoint has no success marker; rerun the Stage 1 audit.")
source_success = load_json(source_success_path)
if (
    source_success.get("stage") != "sources_complete"
    or source_success.get("split_preset") != "gnn-full"
    or source_success.get("quarters") != list(EXPECTED_QUARTERS)
):
    raise RuntimeError("Source success marker does not match the frozen experiment.")
recorded_source_artifacts = source_success.get("artifacts", {})
if recorded_source_artifacts and (
    not isinstance(recorded_source_artifacts, dict)
    or any(
        not (DRIVE_DATA / relative).is_file()
        or (DRIVE_DATA / relative).stat().st_size != size
        for relative, size in recorded_source_artifacts.items()
    )
):
    raise RuntimeError("A source artifact no longer matches the durable checkpoint.")

def clear_interrupted_local_scratch() -> None:
    interim = (LOCAL_DATA / "interim").resolve()
    if interim != Path("/content/tekarx-data/interim"):
        raise RuntimeError(f"Refusing to clean unexpected scratch root: {interim}")
    if not interim.exists():
        return
    candidates = [interim / ".duckdb_temp"]
    for pattern in (
        ".cohort-aggregate-*", ".tabular-features-*",
        ".feature-rescue-work-*", ".*-build-*.duckdb",
        ".drug-dictionary-*.duckdb", ".tabular-features-*.duckdb",
        ".feature-rescue-*.duckdb",
    ):
        candidates.extend(interim.glob(pattern))
    for candidate in dict.fromkeys(candidates):
        if not candidate.exists():
            continue
        resolved = candidate.resolve()
        if interim not in resolved.parents:
            raise RuntimeError(f"Unsafe scratch candidate: {resolved}")
        size = tree_size(candidate) if candidate.is_dir() else candidate.stat().st_size
        print(f"Removing interrupted local scratch: {resolved} ({gibibytes(size):.2f} GiB)")
        if candidate.is_dir():
            shutil.rmtree(candidate)
        else:
            candidate.unlink()


clear_interrupted_local_scratch()

def cumulative_artifacts(stage: str) -> tuple[str, ...]:
    names: list[str] = []
    for candidate, artifacts in BUILD_STAGE_ARTIFACTS.items():
        names.extend(artifacts)
        if candidate == stage:
            return tuple(names)
    raise KeyError(stage)


def valid_build_checkpoint(stage: str, marker_name: str) -> dict[str, object] | None:
    marker = DRIVE_DATA / "processed" / marker_name
    if not marker.is_file():
        return None
    payload = load_json(marker)
    if payload.get("git_sha") != RESOLVED_GIT_SHA:
        print(f"Ignoring {marker.name}: different Git SHA")
        return None
    if payload.get("split_preset") != "gnn-full":
        return None
    expected = cumulative_artifacts("rescue" if stage == "features" else stage)
    recorded = payload.get("artifacts")
    if not isinstance(recorded, dict) or set(recorded) != set(expected):
        return None
    for relative in expected:
        path = DRIVE_DATA / "processed" / relative
        if not path.is_file() or path.stat().st_size != recorded[relative]:
            return None
    return payload


checkpoint_candidates = [
    ("features", "_FEATURES_SUCCESS.json"),
    ("rescue", BUILD_STAGE_MARKERS["rescue"]),
    ("tabular", BUILD_STAGE_MARKERS["tabular"]),
    ("dictionary", BUILD_STAGE_MARKERS["dictionary"]),
    ("cohort", BUILD_STAGE_MARKERS["cohort"]),
]
RESTORED_STAGE = None
RESTORED_ARTIFACTS: tuple[str, ...] = ()
for candidate_stage, marker_name in checkpoint_candidates:
    if valid_build_checkpoint(candidate_stage, marker_name) is not None:
        RESTORED_STAGE = candidate_stage
        RESTORED_ARTIFACTS = cumulative_artifacts(
            "rescue" if candidate_stage == "features" else candidate_stage
        )
        break

interim_datasets = ["faers", "drugcentral"]
if BUILD_DAILYMED:
    interim_datasets.append("dailymed")
copy_pairs = [
    (DRIVE_DATA / "interim" / name, LOCAL_DATA / "interim" / name)
    for name in interim_datasets
]
copy_pairs.append(
    (DRIVE_DATA / "raw" / "drugcentral", LOCAL_DATA / "raw" / "drugcentral")
)
copy_pairs.extend(
    (DRIVE_DATA / "processed" / relative, LOCAL_DATA / "processed" / relative)
    for relative in RESTORED_ARTIFACTS
)
require_local_capacity(copy_pairs, headroom_gib=LOCAL_BUILD_HEADROOM_GIB)

for name in interim_datasets:
    copy_tree_verified(
        DRIVE_DATA / "interim" / name,
        LOCAL_DATA / "interim" / name,
        label=f"{name} interim inputs",
    )
copy_tree_verified(
    DRIVE_DATA / "raw" / "drugcentral",
    LOCAL_DATA / "raw" / "drugcentral",
    label="raw DrugCentral dump",
)
for relative in RESTORED_ARTIFACTS:
    copy_file_verified(
        DRIVE_DATA / "processed" / relative,
        LOCAL_DATA / "processed" / relative,
        label=f"restored {relative}",
    )
assert_faers_scope(LOCAL_DATA, require_complete=True)
print(f"Deepest restored build stage: {RESTORED_STAGE or 'sources only'}")


In [ ]:
BUILD_STAGE_ORDER = tuple(BUILD_STAGE_ARTIFACTS)
restored_rank = (
    len(BUILD_STAGE_ORDER)
    if RESTORED_STAGE == "features"
    else BUILD_STAGE_ORDER.index(RESTORED_STAGE) + 1 if RESTORED_STAGE else 0
)

def checkpoint_build_stage(stage: str) -> None:
    local_processed = LOCAL_DATA / "processed"
    drive_processed = DRIVE_DATA / "processed"
    for relative in BUILD_STAGE_ARTIFACTS[stage]:
        copy_file_verified(
            local_processed / relative, drive_processed / relative,
            label=f"{stage} checkpoint: {relative}",
        )
    artifacts = cumulative_artifacts(stage)
    artifact_sizes: dict[str, int] = {}
    for relative in artifacts:
        local_path = local_processed / relative
        drive_path = drive_processed / relative
        if not local_path.is_file() or not drive_path.is_file():
            raise FileNotFoundError(f"Incomplete {stage} checkpoint: {relative}")
        if local_path.stat().st_size != drive_path.stat().st_size:
            raise RuntimeError(f"Checkpoint size mismatch: {relative}")
        artifact_sizes[relative] = drive_path.stat().st_size
    write_json_atomic(
        drive_processed / BUILD_STAGE_MARKERS[stage],
        {
            "stage": f"{stage}_complete",
            "completed_at_utc": datetime.now(UTC).isoformat(),
            "git_sha": RESOLVED_GIT_SHA,
            "split_preset": "gnn-full",
            "artifacts": artifact_sizes,
        },
    )
    print(f"Durable checkpoint complete: {stage}")


stage_commands = {
    "cohort": ("build-cohort", "--split-preset", "gnn-full"),
    "dictionary": ("build-drug-dictionary",),
    "tabular": ("add-tabular-features", "--skip-graph"),
    "rescue": ("feature-rescue", "--skip-graph"),
}
for stage_rank, stage in enumerate(BUILD_STAGE_ORDER, start=1):
    if stage_rank <= restored_rank:
        print(f"SKIP {stage}: restored from a verified Drive checkpoint")
        continue
    free_gib = gibibytes(shutil.disk_usage("/content").free)
    print(f"Starting {stage}; /content free: {free_gib:.2f} GiB")
    if free_gib < 25:
        raise RuntimeError(f"Refusing {stage}: less than 25 GiB local headroom")
    run_tekarx(
        *stage_commands[stage],
        "--threads", THREADS,
        "--memory-limit", MEMORY_LIMIT,
        data_dir=LOCAL_DATA,
    )
    if stage == "cohort":
        split_path = LOCAL_DATA / "processed" / "splits" / "faers_gnn-full.json"
        if not split_path.is_file():
            write_json_atomic(
                split_path,
                {
                    "dataset": "FAERS",
                    "preset": "gnn-full",
                    "created_at_utc": datetime.now(UTC).isoformat(),
                    "group_key": "CASEID",
                    "version_key": "CASEVERSION",
                    "split_strategy": "quarter-based temporal holdout",
                    "splits": SPLITS,
                },
            )
        print(f"Verified local split definition: {split_path}")
    checkpoint_build_stage(stage)


In [ ]:
cohort_manifest_path = LOCAL_DATA / "processed" / "cohort_manifest.json"
cohort_manifest = load_json(cohort_manifest_path)
if cohort_manifest.get("split_preset") != "gnn-full" or cohort_manifest.get("splits") != SPLITS:
    raise RuntimeError("Cohort manifest does not match the frozen gnn-full split.")
missing_coverage = cohort_manifest.get("quarter_coverage", {}).get("missing", {})
if set(missing_coverage) != set(SPLITS) or any(missing_coverage.values()):
    raise RuntimeError(f"Incomplete cohort quarter coverage: {missing_coverage}")

feature_cohort = LOCAL_DATA / "processed" / "tekarx_cohort_feature_rescue.parquet"
dose_edges = LOCAL_DATA / "processed" / "edges" / "report_drug_dose.parquet"
for required_path in (feature_cohort, dose_edges):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

cohort_sql = feature_cohort.as_posix().replace("'", "''")
from tekarx.transform.duckdb_runtime import configure_duckdb

connection = duckdb.connect()
try:
    configure_duckdb(
        connection, data_dir=LOCAL_DATA, stage="colab-feature-audit",
        memory_limit=MEMORY_LIMIT, threads=THREADS,
    )
    cohort_audit = connection.execute(
        f"""
        SELECT split, count(*) AS rows, count(DISTINCT primaryid) AS primaryids,
               count(DISTINCT caseid) AS caseids, min(quarter) AS first_quarter,
               max(quarter) AS last_quarter,
               count(*) FILTER (WHERE outcome_codes IS NULL) AS no_recorded_outcome,
               count(*) FILTER (WHERE outcome_codes IS NOT NULL AND is_serious = 0)
                   AS documented_nonserious,
               sum(is_serious)::BIGINT AS documented_serious
        FROM read_parquet('{cohort_sql}')
        GROUP BY split ORDER BY split
        """
    ).df()
finally:
    connection.close()
display(cohort_audit)
if set(cohort_audit["split"]) != set(SPLITS):
    raise RuntimeError("Unexpected cohort split labels.")
if not (cohort_audit["rows"] == cohort_audit["primaryids"]).all():
    raise RuntimeError("Cohort is not unique by primaryid.")
if not (cohort_audit["rows"] == cohort_audit["caseids"]).all():
    raise RuntimeError("A caseid crossed versions or appears more than once.")
if cohort_audit["documented_nonserious"].sum() == 0:
    print(
        "LABEL AUDIT: negatives mean no recorded serious OUTC code; they are not "
        "verified-safe reports. Dropping missing OUTC would leave no negative class."
    )

import hashlib

feature_artifacts = cumulative_artifacts("rescue")
feature_artifact_sizes = {
    relative: (DRIVE_DATA / "processed" / relative).stat().st_size
    for relative in feature_artifacts
}
feature_digest = hashlib.sha256(
    json.dumps(feature_artifact_sizes, sort_keys=True).encode("utf-8")
)
for relative in feature_artifacts:
    if relative.endswith("manifest.json"):
        feature_digest.update((DRIVE_DATA / "processed" / relative).read_bytes())
feature_checkpoint_id = feature_digest.hexdigest()
feature_metadata = {
    "stage": "features_complete",
    "completed_at_utc": datetime.now(UTC).isoformat(),
    "git_sha": RESOLVED_GIT_SHA,
    "versions": versions,
    "split_preset": "gnn-full",
    "feature_checkpoint_id": feature_checkpoint_id,
    "test_evaluated": False,
}
write_json_atomic(LOCAL_DATA / "processed" / "colab_run_metadata.json", feature_metadata)
copy_file_verified(
    LOCAL_DATA / "processed" / "colab_run_metadata.json",
    DRIVE_DATA / "processed" / "colab_run_metadata.json",
    label="Colab feature metadata",
)
write_json_atomic(
    DRIVE_DATA / "processed" / "_FEATURES_SUCCESS.json",
    {
        **feature_metadata, "checkpoint": "verified_complete",
        "artifacts": feature_artifact_sizes,
    },
)
print("Feature audit and durable checkpoint complete.")


In [ ]:
feature_cohort = LOCAL_DATA / "processed" / "tekarx_cohort_feature_rescue.parquet"
xgb_device = (
    "cuda"
    if torch.cuda.is_available() and bool(xgboost.build_info().get("USE_CUDA", False))
    else "cpu"
)
print(f"XGBoost baseline device: {xgb_device}", flush=True)
run_tekarx(
    "build-graph",
    "--cohort-path",
    feature_cohort,
    "--graph-storage",
    "memory-mapped",
    "--materialization-batch-size",
    MATERIALIZATION_BATCH_SIZE,
    "--xgb-batch-size",
    XGB_BATCH_SIZE,
    "--xgb-rounds",
    1000,
    "--xgb-early-stopping",
    50,
    "--xgb-max-depth",
    0,
    "--xgb-max-leaves",
    63,
    "--xgb-device",
    xgb_device,
    "--threads",
    THREADS,
    "--memory-limit",
    MEMORY_LIMIT,
    data_dir=LOCAL_DATA,
)


In [ ]:
import hashlib

from tekarx.transform.graph_storage import load_graph_arrays

local_processed = LOCAL_DATA / "processed"
drive_processed = DRIVE_DATA / "processed"
graph_descriptor = local_processed / "tekarx_graph.pt"
graph_manifest_path = local_processed / "graph_manifest.json"
graph_manifest = load_json(graph_manifest_path)
if graph_manifest.get("storage", {}).get("format") != "tekarx.memmap_graph":
    raise RuntimeError("Graph is not using the memory-mapped storage format.")
if "train" not in str(graph_manifest.get("ror_scope", "")).lower():
    raise RuntimeError("Drug ROR was not frozen from training patients.")
if "train" not in str(graph_manifest.get("unknown_vocabulary_scope", "")).lower():
    raise RuntimeError("Unknown-drug vocabulary was not selected from training patients.")
if graph_manifest.get("message_passing", {}).get("held_out_patient_messages_to_shared_drugs") is not False:
    raise RuntimeError("Held-out patient messages can reach shared drug nodes.")
if graph_manifest.get("auxiliary_targets", {}).get("included_in_patient_x") is not False:
    raise RuntimeError("Auxiliary outcome targets leaked into patient features.")

bundle = load_graph_arrays(graph_descriptor, mmap_mode="r")
print(
    {
        name: {"shape": list(array.shape), "dtype": str(array.dtype)}
        for name, array in bundle.arrays.items()
    }
)
print(f"Validation-only XGBoost AUC: {graph_manifest['record']['validation_auc']:.6f}")
del bundle

graph_checkpoint_id = (
    f"graph-{RESOLVED_GIT_SHA[:12]}-"
    f"{datetime.now(UTC).strftime('%Y%m%dT%H%M%S%fZ')}"
)
graph_metadata = {
    **feature_metadata,
    "stage": "graph_complete",
    "completed_at_utc": datetime.now(UTC).isoformat(),
    "graph_storage": "tekarx.memmap_graph",
    "graph_checkpoint_id": graph_checkpoint_id,
    "xgboost_validation_auc": graph_manifest["record"]["validation_auc"],
}
write_json_atomic(local_processed / "colab_run_metadata.json", graph_metadata)

def checkpoint_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


graph_files = (
    "tekarx_graph.pt",
    "graph_manifest.json",
    "tekarx_tabular_baseline.npz",
    "xgboost_baseline.json",
    "colab_run_metadata.json",
)
for name in graph_files:
    if not (local_processed / name).is_file():
        raise FileNotFoundError(local_processed / name)
local_array_dir = local_processed / "tekarx_graph_arrays"
source_sizes = {
    f"tekarx_graph_arrays/{relative}": size
    for relative, size in tree_inventory(local_array_dir).items()
}
source_sizes.update({name: (local_processed / name).stat().st_size for name in graph_files})
control_files = {
    "tekarx_graph.pt",
    "graph_manifest.json",
    "tekarx_graph_arrays/manifest.json",
    "xgboost_baseline.json",
    "colab_run_metadata.json",
}
source_hashes = {
    relative: checkpoint_sha256(
        local_processed / relative
        if not relative.startswith("tekarx_graph_arrays/")
        else local_array_dir / relative.removeprefix("tekarx_graph_arrays/")
    )
    for relative in control_files
}
required_drive_bytes = sum(source_sizes.values()) + 5 * 1024**3
drive_free_bytes = shutil.disk_usage(DRIVE_DATA).free
print(
    f"Graph upload: {gibibytes(sum(source_sizes.values())):.2f} GiB; "
    f"Drive free: {gibibytes(drive_free_bytes):.2f} GiB"
)
if drive_free_bytes < required_drive_bytes:
    raise RuntimeError("Insufficient Drive space for a safe versioned graph upload.")

graph_checkpoint_root = drive_processed / "graph_checkpoints"
graph_checkpoint_root.mkdir(parents=True, exist_ok=True)
staging_checkpoint = graph_checkpoint_root / f".uploading-{graph_checkpoint_id}"
final_checkpoint = graph_checkpoint_root / graph_checkpoint_id
if staging_checkpoint.exists() or final_checkpoint.exists():
    raise RuntimeError(f"Graph checkpoint ID collision: {graph_checkpoint_id}")
staging_checkpoint.mkdir()
try:
    copy_tree_verified(
        local_array_dir, staging_checkpoint / "tekarx_graph_arrays",
        label="versioned memory-mapped graph sidecars to Drive",
    )
    for name in graph_files:
        copy_file_verified(
            local_processed / name, staging_checkpoint / name, label=f"graph {name}"
        )
    if tree_inventory(staging_checkpoint) != source_sizes:
        raise RuntimeError("Versioned graph checkpoint has a file or size mismatch.")
    for relative, expected_hash in source_hashes.items():
        if checkpoint_sha256(staging_checkpoint / relative) != expected_hash:
            raise RuntimeError(f"Graph checkpoint hash mismatch: {relative}")
    staged_bundle = load_graph_arrays(
        staging_checkpoint / "tekarx_graph.pt", mmap_mode="r"
    )
    print(f"Validated {len(staged_bundle.arrays)} staged graph arrays.")
    del staged_bundle
    os.replace(staging_checkpoint, final_checkpoint)
except Exception:
    if staging_checkpoint.is_dir():
        shutil.rmtree(staging_checkpoint)
    raise

artifact_manifest = {
    relative: {
        "size_bytes": size,
        **({"sha256": source_hashes[relative]} if relative in source_hashes else {}),
    }
    for relative, size in source_sizes.items()
}
graph_success_payload = {
    **graph_metadata,
    "checkpoint": "verified_complete",
    "checkpoint_dir": final_checkpoint.relative_to(drive_processed).as_posix(),
    "artifacts": artifact_manifest,
}
write_json_atomic(drive_processed / "_GRAPH_SUCCESS.json", graph_success_payload)
published_graph = load_json(drive_processed / "_GRAPH_SUCCESS.json")
if published_graph != graph_success_payload or tree_inventory(final_checkpoint) != source_sizes:
    raise RuntimeError("Published graph checkpoint did not verify.")

# Keep a graph referenced by the previous GNN until its replacement is verified.
gnn_success_path = drive_processed / "_GNN_SUCCESS.json"
previous_gnn = load_json(gnn_success_path) if gnn_success_path.is_file() else {}
protected_graph_id = previous_gnn.get("graph_checkpoint_id")
protected_ids = {graph_checkpoint_id}
if isinstance(protected_graph_id, str):
    protected_ids.add(protected_graph_id)
for candidate in graph_checkpoint_root.iterdir():
    if candidate.is_dir() and candidate.name not in protected_ids:
        resolved = candidate.resolve()
        if resolved.parent != graph_checkpoint_root.resolve():
            raise RuntimeError(f"Unsafe graph checkpoint cleanup target: {candidate}")
        shutil.rmtree(resolved)
legacy_graph_names = (
    "tekarx_graph_arrays", "tekarx_graph.pt", "graph_manifest.json",
    "tekarx_tabular_baseline.npz", "xgboost_baseline.json",
)
if not gnn_success_path.is_file() or isinstance(protected_graph_id, str):
    for name in legacy_graph_names:
        legacy = drive_processed / name
        if legacy.is_dir():
            if legacy.resolve().parent != drive_processed.resolve():
                raise RuntimeError(f"Unsafe legacy graph cleanup target: {legacy}")
            shutil.rmtree(legacy)
        elif legacy.is_file():
            legacy.unlink()
else:
    print("Preserving the legacy graph until the previous GNN is replaced.")
local_checkpoint_root = local_processed / "graph_checkpoints"
local_checkpoint_root.mkdir(parents=True, exist_ok=True)
local_staging = local_checkpoint_root / f".publishing-{graph_checkpoint_id}"
local_versioned = local_checkpoint_root / graph_checkpoint_id
if local_staging.exists() or local_versioned.exists():
    raise RuntimeError(f"Local graph checkpoint collision: {graph_checkpoint_id}")
local_staging.mkdir()
os.replace(local_array_dir, local_staging / "tekarx_graph_arrays")
os.replace(graph_descriptor, local_staging / "tekarx_graph.pt")
os.replace(local_staging, local_versioned)
print(f"Stage 2 complete: {graph_checkpoint_id} is the verified Drive checkpoint.")


## 3. Train on a Colab CUDA GPU

Start a GPU runtime, then rerun Cells 3–8 (mount, configuration, helpers, pinned checkout, installation, and dependency audit) before running the restore cell below. The restore cell imports its own graph loader, validates the versioned Drive checkpoint by size and control-file hash, and copies only the descriptor and memory-mapped arrays to `/content`. The temporary neighbor matrix is also created on `/content`. Training saves a small atomic state checkpoint to Drive after epoch 1 and every five epochs; rerunning the training cell resumes only when the graph and hyperparameters match.


In [ ]:
import hashlib

from tekarx.transform.graph_storage import load_graph_arrays

drive_processed = DRIVE_DATA / "processed"
local_processed = LOCAL_DATA / "processed"
graph_success_path = drive_processed / "_GRAPH_SUCCESS.json"
if not graph_success_path.is_file():
    raise RuntimeError("Drive graph checkpoint is incomplete; its success marker is missing.")
graph_success = load_json(graph_success_path)
if graph_success.get("graph_storage") != "tekarx.memmap_graph":
    raise RuntimeError("Drive graph checkpoint has the wrong storage format.")
if graph_success.get("git_sha") != RESOLVED_GIT_SHA:
    raise RuntimeError("Drive graph checkpoint was built by a different Git revision.")
feature_success_path = drive_processed / "_FEATURES_SUCCESS.json"
if not feature_success_path.is_file():
    raise RuntimeError("Drive feature checkpoint marker is missing.")
feature_success = load_json(feature_success_path)
if (
    not isinstance(graph_success.get("feature_checkpoint_id"), str)
    or graph_success.get("feature_checkpoint_id") != feature_success.get("feature_checkpoint_id")
):
    raise RuntimeError("Graph checkpoint does not match the current feature artifacts.")
graph_checkpoint_id = graph_success.get("graph_checkpoint_id")
checkpoint_dir = graph_success.get("checkpoint_dir")
recorded_artifacts = graph_success.get("artifacts")
if not isinstance(graph_checkpoint_id, str) or not isinstance(checkpoint_dir, str):
    raise RuntimeError("Drive graph marker is not a versioned checkpoint.")
if not isinstance(recorded_artifacts, dict) or not recorded_artifacts:
    raise RuntimeError("Drive graph marker has no artifact inventory.")
graph_checkpoint_root = (drive_processed / "graph_checkpoints").resolve()
drive_graph_checkpoint = (drive_processed / checkpoint_dir).resolve()
if (
    drive_graph_checkpoint.parent != graph_checkpoint_root
    or drive_graph_checkpoint.name != graph_checkpoint_id
):
    raise RuntimeError("Drive graph marker contains an unsafe checkpoint path.")

def restored_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


drive_inventory = tree_inventory(drive_graph_checkpoint)
if set(drive_inventory) != set(recorded_artifacts):
    raise RuntimeError("Drive graph checkpoint file set differs from its marker.")
for relative, metadata in recorded_artifacts.items():
    if not isinstance(metadata, dict) or metadata.get("size_bytes") != drive_inventory[relative]:
        raise RuntimeError(f"Drive graph size mismatch: {relative}")
    expected_hash = metadata.get("sha256")
    if expected_hash is not None and restored_sha256(drive_graph_checkpoint / relative) != expected_hash:
        raise RuntimeError(f"Drive graph hash mismatch: {relative}")

restore_relatives = {
    relative
    for relative in recorded_artifacts
    if relative == "tekarx_graph.pt" or relative.startswith("tekarx_graph_arrays/")
}
if "tekarx_graph.pt" not in restore_relatives or "tekarx_graph_arrays/manifest.json" not in restore_relatives:
    raise RuntimeError("Drive graph checkpoint lacks its descriptor or array manifest.")
local_graph_root = local_processed / "graph_checkpoints"
local_graph_root.mkdir(parents=True, exist_ok=True)
local_graph_checkpoint = local_graph_root / graph_checkpoint_id
local_staging = local_graph_root / f".restoring-{graph_checkpoint_id}"

def local_graph_is_valid(root: Path) -> bool:
    if not root.is_dir():
        return False
    inventory = tree_inventory(root)
    if set(inventory) != restore_relatives:
        return False
    for relative in restore_relatives:
        metadata = recorded_artifacts[relative]
        if inventory[relative] != metadata["size_bytes"]:
            return False
        expected_hash = metadata.get("sha256")
        if expected_hash is not None and restored_sha256(root / relative) != expected_hash:
            return False
    return True


if not local_graph_is_valid(local_graph_checkpoint):
    for incomplete in (local_graph_checkpoint, local_staging):
        if incomplete.exists():
            resolved = incomplete.resolve()
            if resolved.parent != local_graph_root.resolve():
                raise RuntimeError(f"Unsafe local graph cleanup target: {incomplete}")
            shutil.rmtree(resolved)
    graph_copy_pairs = [
        (
            drive_graph_checkpoint / "tekarx_graph.pt",
            local_staging / "tekarx_graph.pt",
        ),
        (
            drive_graph_checkpoint / "tekarx_graph_arrays",
            local_staging / "tekarx_graph_arrays",
        ),
    ]
    require_local_capacity(graph_copy_pairs, headroom_gib=10)
    local_staging.mkdir()
    try:
        copy_file_verified(
            drive_graph_checkpoint / "tekarx_graph.pt",
            local_staging / "tekarx_graph.pt",
            label="versioned graph descriptor",
        )
        copy_tree_verified(
            drive_graph_checkpoint / "tekarx_graph_arrays",
            local_staging / "tekarx_graph_arrays",
            label="versioned graph arrays to local SSD",
        )
        if not local_graph_is_valid(local_staging):
            raise RuntimeError("Local graph restore failed artifact validation.")
        os.replace(local_staging, local_graph_checkpoint)
    except Exception:
        if local_staging.is_dir():
            shutil.rmtree(local_staging)
        raise
else:
    print(f"Reusing verified local graph checkpoint: {graph_checkpoint_id}")

for candidate in local_graph_root.iterdir():
    if candidate.is_dir() and candidate != local_graph_checkpoint:
        resolved = candidate.resolve()
        if resolved.parent != local_graph_root.resolve():
            raise RuntimeError(f"Unsafe local graph cleanup target: {candidate}")
        shutil.rmtree(resolved)

RESTORED_GRAPH_PATH = local_graph_checkpoint / "tekarx_graph.pt"
restored_bundle = load_graph_arrays(RESTORED_GRAPH_PATH, mmap_mode="r")
print(f"Restored {len(restored_bundle.arrays)} validated graph arrays.")
del restored_bundle


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU, then rerun setup.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"GPU memory: {gibibytes(gpu.total_memory):.2f} GiB")
print(f"Torch CUDA runtime: {torch.version.cuda}")


In [ ]:
drive_processed = DRIVE_DATA / "processed"
local_processed = LOCAL_DATA / "processed"
graph_success = load_json(drive_processed / "_GRAPH_SUCCESS.json")
graph_checkpoint_id = graph_success.get("graph_checkpoint_id")
if not isinstance(graph_checkpoint_id, str):
    raise RuntimeError("Restore the current versioned graph checkpoint before training.")
RESTORED_GRAPH_PATH = (
    local_processed / "graph_checkpoints" / graph_checkpoint_id / "tekarx_graph.pt"
)
if not RESTORED_GRAPH_PATH.is_file():
    raise FileNotFoundError("Rerun the graph-restore cell in this GPU runtime.")
training_checkpoint_dir = drive_processed / "gnn_training_checkpoints"
training_checkpoint_dir.mkdir(parents=True, exist_ok=True)
training_checkpoint_path = training_checkpoint_dir / (
    f"{graph_checkpoint_id}-prospective-seed{GNN_SEED}-b{GNN_BATCH_SIZE}.pt"
)
training_arguments = [
    "train-gnn",
    "--graph-path",
    RESTORED_GRAPH_PATH,
    "--device",
    "cuda",
    "--feature-track",
    "prospective",
    "--batch-size",
    GNN_BATCH_SIZE,
    "--edge-chunk-size",
    EDGE_CHUNK_SIZE,
    "--seed",
    GNN_SEED,
    "--epochs",
    100,
    "--patience",
    15,
    "--checkpoint-path",
    training_checkpoint_path,
    "--checkpoint-every",
    5,
]
if training_checkpoint_path.is_file():
    print(f"Resuming from durable epoch checkpoint: {training_checkpoint_path}")
    training_arguments.extend(("--resume-from", training_checkpoint_path))
else:
    print(f"Starting a new run; epoch checkpoints will be saved to {training_checkpoint_path}")
run_tekarx(*training_arguments, data_dir=LOCAL_DATA)


In [ ]:
import hashlib

local_processed = LOCAL_DATA / "processed"
drive_processed = DRIVE_DATA / "processed"
local_model = local_processed / "tekarx_inductive_gnn.pt"
local_manifest = local_processed / "tekarx_inductive_gnn_manifest.json"
gnn_manifest = load_json(local_manifest)
if gnn_manifest.get("record", {}).get("test_auc") is not None:
    raise RuntimeError("The final test labels were consumed during model selection.")
if gnn_manifest.get("leakage_controls", {}).get("test_evaluated") is not False:
    raise RuntimeError("GNN manifest does not confirm the locked-test protocol.")
if gnn_manifest.get("feature_track") != "prospective":
    raise RuntimeError("Unexpected GNN feature track.")
resumability = gnn_manifest.get("resumability", {})
if resumability.get("checkpoint_test_evaluated") is not False:
    raise RuntimeError("Training checkpoint does not preserve the locked-test protocol.")
if Path(resumability.get("checkpoint_path", "")).resolve() != training_checkpoint_path.resolve():
    raise RuntimeError("GNN manifest references an unexpected epoch checkpoint.")
if not training_checkpoint_path.is_file():
    raise FileNotFoundError("The final resumable epoch checkpoint is missing from Drive.")
if Path(gnn_manifest.get("record", {}).get("graph_path", "")).resolve() != RESTORED_GRAPH_PATH.resolve():
    raise RuntimeError("The trained model does not reference the restored graph checkpoint.")
model_payload = torch.load(local_model, map_location="cpu", weights_only=True)
if not isinstance(model_payload, dict) or model_payload.get("test_auc") is not None:
    raise RuntimeError("Saved model is unreadable or consumed the locked test split.")
del model_payload

graph_success = load_json(drive_processed / "_GRAPH_SUCCESS.json")
graph_checkpoint_id = graph_success.get("graph_checkpoint_id")
if not isinstance(graph_checkpoint_id, str):
    raise RuntimeError("Current graph marker is not versioned.")
if graph_checkpoint_id != RESTORED_GRAPH_PATH.parent.name:
    raise RuntimeError("The Drive graph marker changed during training; refusing to publish.")
gnn_checkpoint_id = (
    f"gnn-{RESOLVED_GIT_SHA[:12]}-"
    f"{datetime.now(UTC).strftime('%Y%m%dT%H%M%S%fZ')}"
)
training_metadata = {
    "stage": "gnn_training_complete",
    "completed_at_utc": datetime.now(UTC).isoformat(),
    "git_sha": RESOLVED_GIT_SHA,
    "versions": versions,
    "gpu": gpu.name,
    "torch_cuda": torch.version.cuda,
    "split_preset": "gnn-full",
    "graph_checkpoint_id": graph_checkpoint_id,
    "gnn_checkpoint_id": gnn_checkpoint_id,
    "epoch_checkpoint": training_checkpoint_path.relative_to(drive_processed).as_posix(),
    "validation_auc": gnn_manifest["record"]["validation_auc"],
    "best_epoch": gnn_manifest["record"]["best_epoch"],
    "test_evaluated": False,
}
write_json_atomic(local_processed / "colab_training_metadata.json", training_metadata)
model_files = (
    "tekarx_inductive_gnn.pt",
    "tekarx_inductive_gnn_manifest.json",
    "colab_training_metadata.json",
)

def model_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


model_artifacts = {
    name: {
        "size_bytes": (local_processed / name).stat().st_size,
        "sha256": model_sha256(local_processed / name),
    }
    for name in model_files
}
model_bytes = sum(item["size_bytes"] for item in model_artifacts.values())
if shutil.disk_usage(DRIVE_DATA).free < model_bytes + 1024**3:
    raise RuntimeError("Insufficient Drive space for a safe versioned model upload.")
gnn_checkpoint_root = drive_processed / "gnn_checkpoints"
gnn_checkpoint_root.mkdir(parents=True, exist_ok=True)
staging_checkpoint = gnn_checkpoint_root / f".uploading-{gnn_checkpoint_id}"
final_checkpoint = gnn_checkpoint_root / gnn_checkpoint_id
if staging_checkpoint.exists() or final_checkpoint.exists():
    raise RuntimeError(f"GNN checkpoint ID collision: {gnn_checkpoint_id}")
staging_checkpoint.mkdir()
try:
    for name in model_files:
        copy_file_verified(
            local_processed / name, staging_checkpoint / name, label=f"GNN {name}"
        )
    if set(tree_inventory(staging_checkpoint)) != set(model_artifacts):
        raise RuntimeError("Staged GNN checkpoint has an unexpected file set.")
    for name, metadata in model_artifacts.items():
        staged = staging_checkpoint / name
        if staged.stat().st_size != metadata["size_bytes"] or model_sha256(staged) != metadata["sha256"]:
            raise RuntimeError(f"GNN checkpoint verification failed: {name}")
    staged_payload = torch.load(
        staging_checkpoint / "tekarx_inductive_gnn.pt",
        map_location="cpu", weights_only=True,
    )
    if not isinstance(staged_payload, dict) or staged_payload.get("test_auc") is not None:
        raise RuntimeError("Staged GNN model failed the locked-test audit.")
    del staged_payload
    os.replace(staging_checkpoint, final_checkpoint)
except Exception:
    if staging_checkpoint.is_dir():
        shutil.rmtree(staging_checkpoint)
    raise

gnn_success_payload = {
    **training_metadata,
    "checkpoint": "verified_complete",
    "checkpoint_dir": final_checkpoint.relative_to(drive_processed).as_posix(),
    "artifacts": model_artifacts,
}
write_json_atomic(drive_processed / "_GNN_SUCCESS.json", gnn_success_payload)
if load_json(drive_processed / "_GNN_SUCCESS.json") != gnn_success_payload:
    raise RuntimeError("Published GNN checkpoint marker did not verify.")

# The replacement marker is durable; old versioned and legacy payloads can now be pruned.
for candidate in gnn_checkpoint_root.iterdir():
    if candidate.is_dir() and candidate.name != gnn_checkpoint_id:
        resolved = candidate.resolve()
        if resolved.parent != gnn_checkpoint_root.resolve():
            raise RuntimeError(f"Unsafe GNN checkpoint cleanup target: {candidate}")
        shutil.rmtree(resolved)
graph_checkpoint_root = drive_processed / "graph_checkpoints"
if graph_checkpoint_root.is_dir():
    for candidate in graph_checkpoint_root.iterdir():
        if candidate.is_dir() and candidate.name != graph_checkpoint_id:
            resolved = candidate.resolve()
            if resolved.parent != graph_checkpoint_root.resolve():
                raise RuntimeError(f"Unsafe graph checkpoint cleanup target: {candidate}")
            shutil.rmtree(resolved)
if training_checkpoint_dir.is_dir():
    for candidate in training_checkpoint_dir.iterdir():
        if candidate.is_file() and candidate != training_checkpoint_path:
            candidate.unlink()
for name in (
    "tekarx_inductive_gnn.pt", "tekarx_inductive_gnn_manifest.json",
    "colab_training_metadata.json", "tekarx_graph_arrays",
    "tekarx_graph.pt", "graph_manifest.json",
    "tekarx_tabular_baseline.npz", "xgboost_baseline.json",
):
    legacy = drive_processed / name
    if legacy.is_dir():
        if legacy.resolve().parent != drive_processed.resolve():
            raise RuntimeError(f"Unsafe legacy cleanup target: {legacy}")
        shutil.rmtree(legacy)
    elif legacy.is_file():
        legacy.unlink()

print(json.dumps(training_metadata, indent=2))
print("Stage 3 complete: validation-selected model safely stored in Drive; test remains locked.")


## Finished

The durable experiment is under `MyDrive/Teka-Rx-full/data/processed/`. `_GRAPH_SUCCESS.json` and `_GNN_SUCCESS.json` point to verified versioned directories under `graph_checkpoints/` and `gnn_checkpoints/`; treat each marker and its referenced directory as one checkpoint. Compare models only with validation metrics until all choices are frozen. Final test evaluation should be a separate, deliberate release step.
